# 05 Final Analysis

Este notebook consolida los hallazgos del proyecto ACV y sirve como cierre tecnico de la Fase 2.


## 1. Carga de resultados

Se leen las metricas ya persistidas en `results/metrics/` para comparar modelos base y tuned.


In [ ]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
metrics_dir = project_root / 'results' / 'metrics'

baseline_cv = pd.read_csv(metrics_dir / 'baseline_model_comparison.csv')
tuned_cv = pd.read_csv(metrics_dir / 'tuned_model_comparison.csv')
before_after = pd.read_csv(metrics_dir / 'before_after_tuning_comparison.csv')
tuned_holdout = pd.read_csv(metrics_dir / 'tuned_holdout_metrics.csv')

baseline_cv, tuned_cv, before_after, tuned_holdout


## 2. Mejor modelo segun Recall y F1

En esta seccion se identifica el modelo mas conveniente para un problema desbalanceado, priorizando la deteccion de positivos sin perder de vista el balance global del `F1`.


In [ ]:
best_recall = tuned_holdout.sort_values(['recall', 'f1', 'roc_auc'], ascending=[False, False, False]).iloc[0]
best_recall


## 3. Interpretacion tecnica

### Hallazgos del EDA y del analisis no supervisado

El analisis exploratorio mostro un dataset con mezcla de variables numericas y categoricas, presencia de nulos ocultos como `Unknown` y un desbalance severo en la clase objetivo (`stroke` cercano al 4.9%). Esto justifico un flujo de limpieza explicito, imputacion controlada, tratamiento de outliers y codificacion robusta dentro de `Pipeline`.

El PCA y el clustering ayudaron a visualizar la estructura general de los datos, pero no mostraron una separacion natural limpia entre positivos y negativos. Eso es consistente con un problema donde la señal predictiva existe, pero no forma grupos perfectamente aislados en baja dimension.

### Comparacion de modelos base y tuned

En los modelos base, `logistic_regression` fue el mejor candidato por `Recall`, `F1` y `ROC-AUC` dentro del holdout. Despues del tuning, el mismo modelo siguio siendo el mas defendible para este problema: mantuvo `Recall` alto (`0.82`), el mejor `F1` del holdout tuned (`0.2303`) y el mejor `ROC-AUC` (`0.8415`).

`svc` mejoro respecto de su baseline y quedo competitivo, mientras que `random_forest` fue el caso mas llamativo: paso de un baseline con muy baja sensibilidad a una version tuned mucho mas agresiva, capaz de recuperar positivos, pero con bastantes falsos positivos.

### Como interpretar la precision baja

La `precision` de los mejores modelos ronda el 12%-14%. Eso no significa que el modelo "acierte solo un 12%". Significa otra cosa: de todos los casos que el modelo marca como ACV, alrededor del 12%-14% realmente lo son. En un dataset donde los positivos reales son apenas el 4.9%, este comportamiento aparece cuando priorizamos `Recall` para no perder casos positivos.

Por eso, en este proyecto no basta mirar una sola metrica. La lectura correcta es:

- `Recall` alto: el modelo logra detectar la mayor parte de los ACV reales.
- `Precision` baja: a cambio, genera bastantes falsas alarmas.
- `F1` moderado: resume ese equilibrio imperfecto entre sensibilidad y precision.
- `ROC-AUC` sobre `0.84`: indica una capacidad discriminativa razonable del ranking de riesgo.

### Justificacion del modelo seleccionado

Se recomienda `logistic_regression` como modelo final porque entrega el mejor balance practico para el objetivo del proyecto:

- sostiene el mejor `Recall` del holdout tuned,
- conserva el mejor `F1` dentro de los modelos evaluados,
- mantiene el mayor `ROC-AUC`,
- y ademas es el modelo mas interpretable para defender tecnicamente en la presentacion.

### Limitaciones del enfoque actual

- El dataset esta fuertemente desbalanceado, lo que limita la precision alcanzable.
- Aun no se hizo ajuste formal del umbral de decision para optimizar el trade-off entre `Recall` y `Precision`.
- El clustering es util como apoyo exploratorio, pero no reemplaza la evidencia supervisada para elegir el modelo final.



## 4. Conclusion final

El proyecto cumple con el ciclo completo solicitado por la pauta: exploracion, limpieza, modelado supervisado y no supervisado, evaluacion comparativa y optimizacion de hiperparametros. La evidencia generada muestra que el problema de prediccion de ACV esta condicionado por un desbalance severo de clases, por lo que la interpretacion correcta debe priorizar `Recall` y `F1` por sobre una lectura ingenua de `Accuracy`.

Dentro de los modelos evaluados, `logistic_regression` queda como la alternativa final mas justificada. No es el modelo con menor cantidad de falsos positivos, pero si el que ofrece el mejor equilibrio entre capacidad de deteccion, estabilidad y defendibilidad tecnica. En otras palabras, es el modelo mas coherente con el objetivo del proyecto: detectar la mayor cantidad posible de casos positivos de ACV sin perder trazabilidad metodologica.

Como mejora futura, el paso mas prometedor seria optimizar el umbral de clasificacion y complementar la evaluacion con curva Precision-Recall, para afinar mejor el equilibrio entre sensibilidad y falsas alarmas.



## Resumen ejecutivo final

- El flujo por fases 0-5 ejecuta correctamente desde `setup_and_run.py --mode run`.
- El proyecto integra EDA, analisis no supervisado, modelado, tuning y evaluacion final.
- El modelo mas defendible es `logistic_regression` balanceada por su capacidad de deteccion positiva y su interpretabilidad.
- Se recomienda su uso como apoyo de screening, no como diagnostico unico.


## Resumen ejecutivo final

Estado de cumplimiento de pauta (Fase 2):

- Flujo por fases 0-5 implementado y ejecutable desde `setup_and_run.py --mode run`.
- Ajuste de hiperparametros con Optuna integrado y comparado contra baseline.
- Reportes finales con metricas de clasificacion y matriz de confusion generados.
- Interpretabilidad incorporada mediante feature importance (y SHAP cuando el entorno lo permite).

Recomendacion final:

- Usar el modelo seleccionado como apoyo de screening, no como diagnostico unico.
- Mantener monitoreo periodico de recall, precision y drift de datos en nuevas corridas.
- Si cambia la distribucion de clases, repetir tuning y recalibracion de umbral.

## Resumen ejecutivo final

Estado de cumplimiento de pauta (Fase 2):

- Flujo por fases 0-5 implementado y ejecutable desde `setup_and_run.py --mode run`.
- Ajuste de hiperparametros con Optuna integrado y comparado contra baseline.
- Reportes finales con metricas de clasificacion y matriz de confusion generados.
- Interpretabilidad incorporada mediante feature importance (y SHAP cuando el entorno lo permite).

Recomendacion final:

- Usar el modelo seleccionado como apoyo de screening, no como diagnostico unico.
- Mantener monitoreo periodico de recall, precision y drift de datos en nuevas corridas.
- Si cambia la distribucion de clases, repetir tuning y recalibracion de umbral.